# Model and Export

## Setup & Imports

In [64]:

# All imports in one place
import json, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor, Pool
from pathlib import Path
import joblib

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def print_metrics(y_true, y_pred, title="Metrics"):
    mae = mean_absolute_error(y_true, y_pred)
    med = median_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{title}:\n  MAE={mae:.2f} | MedAE={med:.2f} | R2={r2:.4f}")

print("Imports OK.")


Imports OK.


## Load Data

In [65]:
DATA_PATH = Path("../data/raw")
raw_path = DATA_PATH / "rents.csv" 

df = pd.read_csv(raw_path)

for c in ['address','district','type']:
    df[c] = df[c].astype('string')
df['total'] = pd.to_numeric(df['total'], errors='coerce')
df.dropna(subset=['total'], inplace=True)


## Train/Test split (stratified by `type`)

In [66]:
features = ['address','district','type','area','bedrooms','garage']
target   = 'total'
RANDOM_STATE = 42

X = df[features].copy()
y = df[target].astype(float).copy()

strat = X['type'].fillna('NA').astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=strat
)

print(f"Original Shapes - Train: {X_train.shape}, Test: {X_test.shape}")

Original Shapes - Train: (9325, 6), Test: (2332, 6)


## Cleaning Data

In [67]:
df_train_temp = X_train.copy()
df_train_temp['total'] = y_train

original_train_count = len(df_train_temp)

print(f"\nOriginal traingin shape: {original_train_count} samples")

area_min = 10
area_max = 1500
df_train_temp = df_train_temp[(df_train_temp['area'] >= area_min) & (df_train_temp['area'] <= area_max)]

preco_m2 = df_train_temp['total'] / df_train_temp['area'].clip(lower=1)
m2_min = 20
m2_max = 400
df_train_temp = df_train_temp[(preco_m2 >= m2_min) & (preco_m2 <= m2_max)]

X_train = df_train_temp[features]
y_train = df_train_temp[target]

print(f"Training shape after cleaning {len(X_train)} samples")

print(f"{original_train_count - len(X_train)} samples removed after cleaning")


Original traingin shape: 9325 samples
Training shape after cleaning 8775 samples
550 samples removed after cleaning


## Preprocessing blocks

In [68]:

num_cols = ['area','bedrooms','garage']
cat_cols = ['address','district','type']

preprocess_ohe = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), cat_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)


## Linear Regression (baseline)

In [69]:

lin_pipe = Pipeline([('preprocess', preprocess_ohe),
                     ('model', LinearRegression())])
lin_pipe.fit(X_train, y_train)
y_pred_lin = lin_pipe.predict(X_test)
print_metrics(y_test, y_pred_lin, title='Linear Regression (baseline)')


Linear Regression (baseline):
  MAE=6279629090041.89 | MedAE=1530.06 | R2=-10556033730095308800.0000


## Random Forest Regressor

In [70]:

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_pipe = Pipeline([('preprocess', preprocess_ohe),
                    ('model', rf)])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
print_metrics(y_test, y_pred_rf, title='Random Forest')


Random Forest:
  MAE=1189.83 | MedAE=702.49 | R2=0.6442


## Linear (Tuned) — Ridge with hyperparameter search

In [71]:

# Build a slightly different preprocessor with scaling on numerics
preprocess_ohe_scale = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), cat_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

ridge_pipe = Pipeline([('preprocess', preprocess_ohe_scale),
                       ('model', Ridge(random_state=RANDOM_STATE))])

param_grid = {
    'model__alpha': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
}

gs = GridSearchCV(ridge_pipe, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=5, n_jobs=-1)
gs.fit(X_train, y_train)

ridge_best = gs.best_estimator_
y_pred_ridge = ridge_best.predict(X_test)
print("Best alpha:", gs.best_params_)
print_metrics(y_test, y_pred_ridge, title='Ridge (tuned)')


Best alpha: {'model__alpha': 3.0}
Ridge (tuned):
  MAE=1350.79 | MedAE=905.27 | R2=0.6009


## CatBoost (predict price per m²)

In [72]:
X_train_cb = X_train_clean.copy()
X_test_cb = X_test.copy() 

# Feature Engineering
X_train_cb['area_log'] = np.log1p(X_train_cb['area'])
X_test_cb['area_log']  = np.log1p(X_test_cb['area'])
X_train_cb['area_per_bedroom'] = X_train_cb['area'] / np.clip(X_train_cb['bedrooms'].astype(float), 1, None)
X_test_cb['area_per_bedroom']  = X_test_cb['area']  / np.clip(X_test_cb['bedrooms'].astype(float), 1, None)
X_train_cb['district_type'] = X_train_cb['district'].astype(str) + '_' + X_train_cb['type'].astype(str)
X_test_cb['district_type']  = X_test_cb['district'].astype(str) + '_' + X_test_cb['type'].astype(str)

# Defining colums for the model
feat_cols_cb = ['area','bedrooms','garage','area_log','area_per_bedroom','type','district', 'address', 'district_type']
cat_cols_cb  = ['type','district', 'address', 'district_type']

# 3. Train/Test split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_cb, y_train_clean, test_size=0.2, random_state=RANDOM_STATE, stratify=X_train_cb['type'].astype(str)
)

# Transforming the target
y_tr_m2_log  = np.log1p(y_tr / np.clip(X_tr['area'].astype(float), 1, None))
y_val_m2_log = np.log1p(y_val / np.clip(X_val['area'].astype(float), 1, None))

# 5. Creating Pools
train_pool = Pool(X_tr[feat_cols_cb], label=y_tr_m2_log, cat_features=cat_cols_cb)
val_pool   = Pool(X_val[feat_cols_cb], label=y_val_m2_log, cat_features=cat_cols_cb)
test_pool  = Pool(X_test_cb[feat_cols_cb], cat_features=cat_cols_cb)

## Training and Prediction

In [73]:
cbr = CatBoostRegressor(
    loss_function='MAE', eval_metric='MAE',
    learning_rate=0.04, 
    depth=8,
    l2_leaf_reg=3.0,
    iterations=5000,
    random_seed=RANDOM_STATE,
    verbose=200,
    od_type='Iter',
    od_wait=300 
)

print("Iniciando o treinamento do CatBoost otimizado...")
cbr.fit(train_pool, eval_set=val_pool)


pred_m2_log_test = cbr.predict(test_pool)

pred_m2_test = np.expm1(pred_m2_log_test)
y_pred_cb = np.clip(pred_m2_test, 0, None) * X_test_cb['area'].astype(float).values

print("\nMétricas do modelo otimizado:")
print_metrics(y_test, y_pred_cb, title='CatBoost Otimizado')

Iniciando o treinamento do CatBoost otimizado...
0:	learn: 0.3869024	test: 0.3963284	best: 0.3963284 (0)	total: 98.7ms	remaining: 8m 13s
200:	learn: 0.2062093	test: 0.2241066	best: 0.2241066 (200)	total: 19.1s	remaining: 7m 35s
400:	learn: 0.1883143	test: 0.2214298	best: 0.2214042 (398)	total: 37.9s	remaining: 7m 15s
600:	learn: 0.1765346	test: 0.2216088	best: 0.2213046 (403)	total: 57.9s	remaining: 7m 3s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.2213045881
bestIteration = 403

Shrink model to first 404 iterations.

Métricas do modelo otimizado:
CatBoost Otimizado:
  MAE=1057.79 | MedAE=549.65 | R2=0.6809


In [74]:
y_pred_ensemble = (y_pred_rf + y_pred_cb) / 2
print("\n--- Ensemble Result: ---")
print_metrics(y_test, y_pred_ensemble, title='Ensemble (RF + CatBoost)')


--- Ensemble Result: ---
Ensemble (RF + CatBoost):
  MAE=1075.41 | MedAE=598.65 | R2=0.6864


## Side-by-side comparison

In [75]:

results = []
results.append(('Linear', mean_absolute_error(y_test, y_pred_lin)))
results.append(('RandomForest', mean_absolute_error(y_test, y_pred_rf)))
results.append(('Ridge (tuned)', mean_absolute_error(y_test, y_pred_ridge)))
results.append(('CatBoost m²', mean_absolute_error(y_test, y_pred_cb)))
pd.DataFrame(results, columns=['model','MAE']).sort_values('MAE')


,model,MAE
3,CatBoost m²,1.057790e+03
1,RandomForest,1.189834e+03
2,Ridge (tuned),1.350791e+03
0,Linear,6.279629e+12


## Error Analysis

In [76]:
df_analise = X_test.copy()
df_analise['real_total'] = y_test
df_analise['predicted_error'] = y_pred_cb
df_analise['error_abs'] = abs(df_analise['real_total'] - df_analise['predicted_error'])

worst_errors = df_analise.sort_values(by='error_abs', ascending=False)

# Exiba os 20 piores erros
print("Analysing the 20 worst errors:")
worst_errors.head(20)

Analysing the 20 worst errors:


,address,district,type,area,bedrooms,garage,real_total,predicted_error,error_abs
6095,Avenida Chibarás,Planalto Paulista,Studio e kitnet,24,1,0,26710.0,1905.935673,24804.064327
7254,Rua Doutor Miranda de Azevedo,Vila Anglo Brasileira,Apartamento,110,2,2,17340.0,6136.735913,11203.264087
6868,Rua Doutor Nicolau de Sousa Queirós,Vila Mariana,Apartamento,85,2,2,17190.0,6079.977857,11110.022143
9458,Avenida Giovanni Gronchi,Vila Andrade,Apartamento,455,4,4,9789.0,20875.849927,11086.849927
8685,Rua Ajururés,Brooklin Paulista,Casa,169,3,2,15930.0,5012.065141,10917.934859
10007,Rua Osório Duque Estrada,Paraíso,Casa,242,4,2,17840.0,6966.708329,10873.291671
10237,Avenida Praia Grande,City Bussocaba,Casa,560,6,6,6483.0,16881.379714,10398.379714
8513,Rua Major Quedinho,Centro,Apartamento,150,2,2,18140.0,7743.878000,10396.122000
9235,Rua Kohei Yokoyana,Jardim Peri Peri,Casa,500,3,6,5049.0,15241.414861,10192.414861
2706,Rua Padre João Manuel,Cerqueira César,Apartamento,100,3,2,18500.0,8484.374235,10015.625765


## Export best model

In [77]:



MODEL_VERSION = '1'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
MODELS_DIR   = (PROJECT_ROOT / 'models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / f'model_v{MODEL_VERSION}.joblib'
META_PATH  = MODELS_DIR / f'model_v{MODEL_VERSION}.json'

# Choose which to export: 'linear', 'rf', 'ridge', 'catboost'
BEST = 'catboost'

if BEST == 'linear':
    to_save = {'model_type': 'sklearn_pipeline', 'model': lin_pipe}
elif BEST == 'rf':
    to_save = {'model_type': 'sklearn_pipeline', 'model': rf_pipe}
elif BEST == 'ridge':
    to_save = {'model_type': 'sklearn_pipeline', 'model': ridge_best}
elif BEST == 'catboost':
    to_save = {'model_type': 'catboost_m2', 'features': feat_cols_cb, 'cat_cols': ['type','district'], 'model': cbr}
else:
    raise ValueError('Unknown BEST model key.')

joblib.dump(to_save, MODEL_PATH)

# metadata
area_p99 = float(np.nanpercentile(df['area'].astype(float), 99))
meta = {'model_version': MODEL_VERSION, 'preprocess': {'clip_area_p99': area_p99}}
META_PATH.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'✅ Saved to: {MODEL_PATH}\n✅ Metadata: {META_PATH}')


✅ Saved to: c:\DEV\brazil-rent-price-estimator\models\model_v1.joblib
✅ Metadata: c:\DEV\brazil-rent-price-estimator\models\model_v1.json
